In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_0.pickle

In [ ]:
%%PandasProfile
### cell 23 ###

def grab_subset_of_data_35(original_df, question_of_interest):
    return (
        original_df
            .filter(like=question_of_interest, axis=1)
            .rename(columns=lambda c: c.rsplit('-', 1)[-1].lstrip())
            .dropna(how='all')
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest, include_2017=False):
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs = [
        responses_df_2018_cell10,
        responses_df_2019_cell10,
        responses_df_2020,
        responses_df_2021,
        responses_df_2022_cell10
    ]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)

    # Vectorized concatenation with year keys
    year_df_map = dict(zip(years, dfs))
    combined = (
        pd.concat(year_df_map, axis=0, names=['year'])
          .filter(like=question_of_interest, axis=1)
          .rename(columns=lambda c: c.rsplit('-', 1)[-1].lstrip())
          .dropna(how='all')
          .reset_index(level='year')
          .reset_index(drop=True)
    )
    counts = (
        combined
          .groupby('year')
          .count()
          .reindex(years, fill_value=0)
          .reset_index()
    )
    return combined, counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Compute respondent totals once and broadcast
    totals = df['year'].value_counts().reindex(df_counts['year']).fillna(0)
    return (
        df_counts
          .set_index('year')
          .div(totals, axis=0)
          .mul(100)
          .reset_index()
    )

# Main pipeline
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'
language_df_combined, language_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined,
    language_df_combined_counts
)

# Select & reshape
langs = ['year', 'Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']
language_df_combined_percentages_cell35 = language_df_combined_percentages[langs]

df_cell35 = (
    language_df_combined_percentages_cell35
      .melt(id_vars='year', value_vars=langs[1:])
      .sort_values(['year', 'value'])
      .rename(columns={'variable': ''})
)

df_cell35.info()